In [39]:
!pip install pykeen -q

In [40]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes

In [41]:
import warnings
warnings.filterwarnings("ignore")

import os, json, csv, time, pickle, torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from collections import defaultdict, deque, Counter
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
import torch, pickle 

In [42]:
import json, os

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

# Peek at entities.json structure
with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)

# See what it looks like
first_keys = list(ent_json.keys())[:3]
for k in first_keys:
    print(k, "→", ent_json[k])

# Peek at one txt file
sample_txt = f"{BASE}/data/entities/en/extracts/extracts/Q100.txt"
with open(sample_txt) as f:
    print("\nQ100.txt:\n", f.read()[:300])

Q202042 → {'wiki': 'https://en.wikipedia.org/wiki/Euskaltzaindia', 'description': 'official academic language regulatory institution which watches over the Basque language', 'label': 'Euskaltzaindia'}
Q113400 → {'wiki': 'https://en.wikipedia.org/wiki/Fritz_Glatz', 'description': 'Austrian racing driver', 'label': 'Fritz Glatz'}
Q432602 → {'wiki': 'https://en.wikipedia.org/wiki/David_Hidalgo', 'description': 'American musician', 'label': 'David Hidalgo'}

Q100.txt:
 Boston (UK: , US: ) is the capital and most populous city of the Commonwealth of Massachusetts in the United States, and the 21st most populous city in the United States. The city proper covers 49 square miles (127 km2) with an estimated population of 694,583 in 2018, also making it the most populou


In [43]:
import json, os, pandas as pd

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"


with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)   # { "Q202042": {"label": "Euskaltzaindia", ...}, ... }

with open(f"{BASE}/data/relations/en/relations.json") as f:
    rel_json = json.load(f)   # same nested structure for relations

def ent(qid): return ent_json.get(qid, {}).get("label", qid)
def rel(pid): return rel_json.get(pid, {}).get("label", pid)

def load_split(split_name):
    path = f"{BASE}/data/triples/codex-s/{split_name}.txt"
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3:
                h, r, t = parts
                rows.append((ent(h), rel(r), ent(t)))
    return pd.DataFrame(rows, columns=["head", "relation", "tail"])

df_train = load_split("train")
df_test  = load_split("test")
df_valid = load_split("valid")

print(f"Train: {len(df_train)}  Valid: {len(df_valid)}  Test: {len(df_test)}")
print(df_train.head(10))

def read_id_list(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

entity_qids   = sorted(ent_json.keys())
relation_pids = sorted(rel_json.keys())

entity_to_id   = {ent(qid): i for i, qid in enumerate(entity_qids)}
relation_to_id = {rel(pid): i for i, pid in enumerate(relation_pids)}
id_to_entity   = {i: ent(qid) for i, qid in enumerate(entity_qids)}
id_to_relation = {i: rel(pid) for i, pid in enumerate(relation_pids)}

print(f"Entities: {len(entity_to_id)}   Relations: {len(relation_to_id)}")
print(id_to_entity[0], "|", id_to_entity[1], "|", id_to_entity[2])
print(id_to_relation[0], "|", id_to_relation[1])

Train: 32888  Valid: 1827  Test: 1828
                           head                              relation  \
0                Leonhard Euler  languages spoken, written, or signed   
1                 Carl Djerassi                        cause of death   
2                      Colombia                             member of   
3  Aleksey Nikolayevich Tolstoy  languages spoken, written, or signed   
4                   Netherlands                             member of   
5                 Henry Rollins                            occupation   
6                Richard Wagner                            occupation   
7                   Jaden Smith                            occupation   
8               La Toya Jackson                               sibling   
9         John Cameron Mitchell                            occupation   

                                tail  
0                             German  
1                             cancer  
2  International Finance Corporation  
3 

In [44]:
df_train.to_csv("CODEX_S_train.csv",index = None)
df_test.to_csv("CODEX_S_test.csv",index = None)
df_valid.to_csv("CODEX_S_valid.csv",index = None)

In [45]:
!git clone https://github.com/uma-pi1/kge.git

fatal: destination path 'kge' already exists and is not an empty directory.


In [46]:
import sys

# CLEAN everything related to kge
sys.modules.pop("kge", None)

# REMOVE wrong paths
sys.path = [p for p in sys.path if "kge" not in p]

# ADD ONLY THIS
sys.path.insert(0, "/kaggle/working/kge")

In [47]:
import kge
print(kge.__file__)

/kaggle/working/kge/kge/__init__.py


In [48]:
import sys
sys.modules.pop("kge", None)

<module 'kge' from '/kaggle/working/kge/kge/__init__.py'>

In [49]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_ax" in line:
        continue  # REMOVE the line entirely
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed Ax import completely")

Removed Ax import completely


In [50]:
!rm /kaggle/working/kge/kge/job/search_ax.py

rm: cannot remove '/kaggle/working/kge/kge/job/search_ax.py': No such file or directory


In [51]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_grash" in line:
        continue  # remove this import
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed GraSH import")

Removed GraSH import


In [52]:
from kge.model import KgeModel
from kge.util.io import load_checkpoint

print("Import worked")

Import worked


In [53]:
MODEL_PATH = r"/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt"
print(MODEL_PATH)


/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt


In [54]:
file_path = "/kaggle/working/kge/kge/util/io.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(checkpoint_file, map_location=\"cpu\")",
    "torch.load(checkpoint_file, map_location=\"cpu\", weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched torch.load")

Patched torch.load


In [55]:
import os

os.environ["KGE_DATASET_DIR"] = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master/data/triples"

In [56]:
import torch
from kge.config import Config

torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

In [57]:
import sys
import torch

sys.path.insert(0, "/kaggle/working")

from kge.config import Config
from kge.util.io import load_checkpoint

# PyTorch safety fix
torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

# reload checkpoint
checkpoint = load_checkpoint(MODEL_PATH, device="cpu")

print("Checkpoint loaded")

Checkpoint loaded


In [58]:
from pykeen.triples import TriplesFactory

In [59]:
import torch, json, pickle, numpy as np
from pathlib import Path

MODEL_DIR  = "/kaggle/input/datasets/aaryaupi/model-libkge-kaggle-upload/kaggle_upload"
CODEX_BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

CKPT_PATH     = f"{MODEL_DIR}/winner_model.pt"
ENTITY_IDS    = f"{MODEL_DIR}/entity_ids.del"
RELATION_IDS  = f"{MODEL_DIR}/relation_ids.del"

import torch

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt["model"][0]

entity_emb   = state_dict["_base_model._entity_embedder._embeddings.weight"].detach().float()
relation_emb = state_dict["_base_model._relation_embedder._embeddings.weight"].detach().float()

print(f"entity_emb:   {entity_emb.shape}")
print(f"relation_emb: {relation_emb.shape}")
print(f"trained epoch: {ckpt['epoch']}")

# Fix here
best_valid = ckpt["valid_trace"][-1]
print(f"best valid MRR: {best_valid['mean_reciprocal_rank_filtered_with_test']:.4f}")

entity_emb:   torch.Size([2034, 512])
relation_emb: torch.Size([84, 512])
trained epoch: 295
best valid MRR: 0.4742


In [60]:

def load_del(path):
    id_to_qid = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                id_to_qid[int(parts[0])] = parts[1]
    return id_to_qid

id_to_entity_qid   = load_del(ENTITY_IDS)
id_to_relation_pid = load_del(RELATION_IDS)
entity_qid_to_id   = {v: k for k, v in id_to_entity_qid.items()}
relation_pid_to_id = {v: k for k, v in id_to_relation_pid.items()}


with open(f"{CODEX_BASE}/data/entities/en/entities.json")  as f: ent_json = json.load(f)
with open(f"{CODEX_BASE}/data/relations/en/relations.json") as f: rel_json = json.load(f)

def elabel(qid): return ent_json.get(qid, {}).get("label", qid)
def rlabel(pid): return rel_json.get(pid, {}).get("label", pid)

id_to_entity_label   = {i: elabel(q) for i, q in id_to_entity_qid.items()}
id_to_relation_label = {i: rlabel(p) for i, p in id_to_relation_pid.items()}
label_to_entity_id   = {v: k for k, v in id_to_entity_label.items()}
label_to_relation_id = {v: k for k, v in id_to_relation_label.items()}

print(f"Entities:  {len(id_to_entity_label)}")
print(f"Relations: {len(id_to_relation_label)}")
print(f"\nAll relations:")
for i, label in sorted(id_to_relation_label.items()):
    print(f"  {i:3d}: {label}")

Entities:  2034
Relations: 42

All relations:
    0: languages spoken, written, or signed
    1: cause of death
    2: member of
    3: occupation
    4: sibling
    5: diplomatic relation
    6: continent
    7: ethnic group
    8: country of citizenship
    9: employer
   10: educated at
   11: place of birth
   12: influenced by
   13: field of work
   14: genre
   15: record label
   16: instrument
   17: official language
   18: unmarried partner
   19: place of death
   20: country
   21: religion
   22: movement
   23: residence
   24: spouse
   25: member of political party
   26: place of burial
   27: part of
   28: medical condition
   29: child
   30: time period
   31: headquarters location
   32: narrative location
   33: location of formation
   34: head of state
   35: named after
   36: parent organization
   37: notable works
   38: cast member
   39: founded by
   40: country of origin
   41: practiced by


In [61]:
def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key   # dict: key -> row_idx
    values        = kvs_index._values         # 1D LongTensor, all values concatenated
    offsets        = kvs_index._values_offset  # 1D LongTensor, len = n_keys + 1

    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f:
    train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl", "rb") as f:
    test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl", "rb") as f:
    test_po_to_s  = kvs_to_dict(pickle.load(f))

print("Filter indexes loaded")

Filter indexes loaded


In [62]:
with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    idx = pickle.load(f)

# Inspect what's available
print(type(idx))
print([a for a in dir(idx) if not a.startswith("__")])

<class 'kge.indexing.KvsAllIndex'>
['_convert_key_to_correct_type', '_create_index_of_key_dict', '_get_all_impl', '_index_of_key', '_keys', '_values', '_values_of', '_values_offset', 'default_factory', 'get', 'get_all', 'items', 'key_cols', 'keys', 'sort_triples_by_keys', 'value_col', 'values']


In [63]:

DIM = entity_emb.shape[1] // 2  # 256

def _re(e): return e[..., :DIM]
def _im(e): return e[..., DIM:]

def score_triple(s, p, o):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (torch.sum(sr*pr*or_) - torch.sum(si*pi*or_) +
            torch.sum(sr*pi*oi) + torch.sum(si*pr*oi)).item()

def _scores_sp(s, p):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    return (_re(entity_emb) @ (sr*pr - si*pi) +
            _im(entity_emb) @ (sr*pi + si*pr))

def _scores_po(p, o):
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (_re(entity_emb) @ (pr*or_ - pi*oi) +
            _im(entity_emb) @ (pr*oi + pi*or_))

def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key
    values       = kvs_index._values
    offsets      = kvs_index._values_offset
    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f: train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f: train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl",  "rb") as f: test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl",  "rb") as f: test_po_to_s  = kvs_to_dict(pickle.load(f))
print("Filter indexes loaded")


def evaluate(split="test"):
    path = f"{CODEX_BASE}/data/triples/codex-s/{split}.txt"
    rr_list, hits1_list, hits10_list = [], [], []

    with open(path) as f:
        triples = [l.strip().split("\t") for l in f if l.strip()]

    for h_qid, r_pid, t_qid in triples:
        s = entity_qid_to_id.get(h_qid)
        p = relation_pid_to_id.get(r_pid)
        o = entity_qid_to_id.get(t_qid)
        if None in (s, p, o):
            continue

        scores = _scores_sp(s, p).clone()
        known  = train_sp_to_o.get((s, p), set()) | test_sp_to_o.get((s, p), set())
        known.discard(o)
        for k in known:
            scores[k] = float("-inf")

        rank = (scores > scores[o]).sum().item() + 1
        rr_list.append(1.0 / rank)
        hits1_list.append(1 if rank == 1 else 0)
        hits10_list.append(1 if rank <= 10 else 0)

    mrr    = float(np.mean(rr_list))
    hits1  = float(np.mean(hits1_list))
    hits10 = float(np.mean(hits10_list))
    print(f"\n── {split.upper()} SET ──────────────────────────────")
    print(f"  MRR:     {mrr:.4f}  (paper: 0.465)")
    print(f"  Hits@1:  {hits1:.4f}  (paper: 0.372)")
    print(f"  Hits@10: {hits10:.4f}  (paper: 0.646)")
    return mrr, hits1, hits10


evaluate("test")

Filter indexes loaded

── TEST SET ──────────────────────────────
  MRR:     0.5759  (paper: 0.465)
  Hits@1:  0.4294  (paper: 0.372)
  Hits@10: 0.8714  (paper: 0.646)


(0.575868337490121, 0.42943107221006566, 0.8714442013129103)

In [64]:
def build_graph_from_df(df):
    graph = defaultdict(list)
    for _, row in df.iterrows():
        graph[row["head"]].append((row["relation"], row["tail"]))
    return graph

graph = build_graph_from_df(df_train)
print(f"Graph nodes: {len(graph)}")

Graph nodes: 1702


In [65]:
EMBS_CACHE = None

def get_entity_embeddings():
    """Return real-valued entity embedding matrix for similarity."""
    embs = entity_emb.float()
    # ComplEx: collapse complex dims by taking magnitude per pair
    re = embs[:, :DIM]
    im = embs[:, DIM:]
    return torch.sqrt(re**2 + im**2)   # shape: (n_entities, DIM)

# ── Subgraph helpers ──────────────────────────────────────────
def extract_subgraph(entity, graph, hops=2, max_triples=150):
    subgraph = []
    visited  = {entity}
    queue    = deque([(entity, 0)])
    while queue and len(subgraph) < max_triples:
        node, depth = queue.popleft()
        if depth >= hops:
            continue
        for rel_label, tail in graph.get(node, []):
            if len(subgraph) >= max_triples:
                break
            subgraph.append((node, rel_label, tail))
            if tail not in visited:
                visited.add(tail)
                queue.append((tail, depth + 1))
    return subgraph

def focused_subgraph(entities, graph, hops=2, max_triples=100, query_relation=None):
    entity_set  = set(entities)
    all_triples = []
    seen        = set()
    for ent_node in entities:
        for triple in extract_subgraph(ent_node, graph, hops, max_triples):
            if triple not in seen:
                seen.add(triple)
                all_triples.append(triple)
    if len(all_triples) <= max_triples:
        return all_triples
    tier1, tier2, tier3, tier4 = [], [], [], []
    for h, r, t in all_triples:
        if query_relation and r == query_relation:       tier1.append((h, r, t))
        elif h in entity_set and t in entity_set:        tier2.append((h, r, t))
        elif h in entity_set or  t in entity_set:        tier3.append((h, r, t))
        else:                                             tier4.append((h, r, t))
    return (tier1 + tier2 + tier3 + tier4)[:max_triples]

def hop_classifier(head, tail, graph, target_relation=None):
    for relation, t in graph.get(head, []):
        if t == tail:
            if target_relation is None or relation == target_relation:
                return "single", 1, relation
    direct_wrong = [r for r, t in graph.get(head, []) if t == tail]
    if direct_wrong:
        return "multi", 1.5, f"direct but via {direct_wrong[0]}"
    paths_found = []
    for r1, mid in graph.get(head, []):
        for r2, t2 in graph.get(mid, []):
            if t2 == tail:
                paths_found.append(f"{head}-{r1}->{mid}-{r2}->{tail}")
    if paths_found:
        return "multi", 2, paths_found[0]
    return "none", 99, []

In [66]:
def get_entity_relations(entity, df):
    head_rels = set(df[df["head"] == entity]["relation"])
    tail_rels = set(df[df["tail"] == entity]["relation"])
    return head_rels | tail_rels

def similarity_summary(entity, k=5):
    e_id  = label_to_entity_id[entity]
    e_vec = EMBS_CACHE[e_id]
    sims  = F.cosine_similarity(e_vec.unsqueeze(0), EMBS_CACHE).detach().cpu().numpy()
    ranked = np.argsort(sims)[::-1]
    entity_rels = get_entity_relations(entity, df_train)
    results = []
    for idx in ranked:
        name = id_to_entity_label[idx]
        if name == entity:
            continue
        score = sims[idx]
        neighbor_rels = get_entity_relations(name, df_train)
        shared    = len(entity_rels & neighbor_rels)
        total     = len(entity_rels | neighbor_rels)
        rel_score = shared / total if total > 0 else 0.0
        results.append((name, score, shared, rel_score))
        if len(results) == k:
            break
    parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})"
               for n, s, sh, rs in results]
    summary = f"{entity} most similar to: {', '.join(parts)}"
    return summary, results

In [67]:
def get_full_ranking_filtered_batched(head, relation, true_tail):
    """
    Score ALL tail candidates using raw ComplEx embeddings.
    Replaces winner_model.score_hrt() entirely.
    """
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    o = label_to_entity_id.get(true_tail)

    if None in (s, p, o):
        raise ValueError(f"Unknown entity/relation: {head}, {relation}, {true_tail}")

    with torch.no_grad():
        all_scores = _scores_sp(s, p).cpu()   # shape: (n_entities,)

    # exclude head itself from ranking
    head_id = label_to_entity_id[head]
    all_scores[head_id] = float("-inf")

    scores_list = all_scores.tolist()
    ranked = sorted(
        [(id_to_entity_label[i], scores_list[i]) for i in range(len(scores_list))
         if scores_list[i] != float("-inf")],
        key=lambda x: -x[1]
    )

    ranked_entities = [e for e, _ in ranked]
    true_rank       = ranked_entities.index(true_tail) + 1

    return {
        "predicted":       ranked[0][0],
        "predicted_score": ranked[0][1],
        "true_tail":       true_tail,
        "true_score":      scores_list[o],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(sc, 3)) for e, sc in ranked],
    }


In [68]:
def get_full_ranking_filtered(head, relation, true_tail, model):
    """
    Same as before but filters self-loop at rank 1.
    In real routing you'd never route a problem to itself.
    """
    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    
    all_scores = []
    for i in range(len(entity_to_id)):
        # skip self-prediction
        if id_to_entity[i] == head:
            continue
        score = model.score_hrt(
            torch.tensor([[h_id, r_id, i]])
        ).item()
        all_scores.append((id_to_entity[i], score, i))
    
    all_scores.sort(key=lambda x: -x[1])
    
    ranked_entities = [e for e, s, i in all_scores]
    true_rank       = ranked_entities.index(true_tail) + 1
    predicted       = all_scores[0][0]
    
    return {
        "predicted":       predicted,
        "predicted_score": all_scores[0][1],
        "true_tail":       true_tail,
        "true_score":      [s for e,s,i in all_scores 
                           if e == true_tail][0],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(s,3)) 
                           for e,s,i in all_scores],
    }

In [69]:
def build_type_constraints(df):
    rel_to_tail_counts = defaultdict(lambda: defaultdict(int))
    rel_to_head_counts = defaultdict(lambda: defaultdict(int))
    rel_to_pair_counts = defaultdict(lambda: defaultdict(int))
    for _, row in df.iterrows():
        h, r, t = row["head"], row["relation"], row["tail"]
        rel_to_tail_counts[r][t] += 1
        rel_to_head_counts[r][h] += 1
        rel_to_pair_counts[r][(h, t)] += 1
    rel_head_to_ranked_tails = defaultdict(dict)
    for r, pair_counts in rel_to_pair_counts.items():
        head_to_tails = defaultdict(dict)
        for (h, t), count in pair_counts.items():
            head_to_tails[h][t] = count
        for h, tail_counts in head_to_tails.items():
            total = sum(tail_counts.values())
            rel_head_to_ranked_tails[(r, h)] = {
                t: round(c / total, 4)
                for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
            }
    rel_to_tail_dist = {}
    for r, tail_counts in rel_to_tail_counts.items():
        total = sum(tail_counts.values())
        rel_to_tail_dist[r] = {
            t: round(c / total, 4)
            for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
        }
    return {
        "rel_head_to_ranked_tails": dict(rel_head_to_ranked_tails),
        "rel_to_tail_dist":         rel_to_tail_dist,
        "rel_to_tail_counts":       dict(rel_to_tail_counts),
        "rel_to_head_counts":       dict(rel_to_head_counts),
    }

def get_type_constraint_signal(head, relation, true_tail, predicted, constraints):
    rh_tails       = constraints["rel_head_to_ranked_tails"]
    tail_dist      = constraints["rel_to_tail_dist"]
    specific       = rh_tails.get((relation, head), {})
    general        = tail_dist.get(relation, {})
    type_fit_true  = specific.get(true_tail) or general.get(true_tail, 0.0)
    type_fit_pred  = specific.get(predicted) or general.get(predicted, 0.0)
    expected_tails = list(specific.keys())[:5] or list(general.keys())[:5]
    tail_counts    = constraints["rel_to_tail_counts"]
    true_rels = {r for r, tails in tail_counts.items() if true_tail in tails}
    pred_rels = {r for r, tails in tail_counts.items() if predicted  in tails}
    return {
        "type_fit_true":  type_fit_true,
        "type_fit_pred":  type_fit_pred,
        "type_gap":       round(type_fit_true - type_fit_pred, 4),
        "expected_tails": expected_tails,
        "only_true_has":  sorted(true_rels - pred_rels),
        "only_pred_has":  sorted(pred_rels - true_rels),
        "shared_rels":    sorted(true_rels & pred_rels),
    }


In [70]:
def failure_summary(head, relation, true_tail, predicted_tail, constraints):
    """Uses score_triple() with raw embeddings — no model object needed."""
    s  = label_to_entity_id[head]
    p  = label_to_relation_id[relation]
    o  = label_to_entity_id[true_tail]
    p2 = label_to_entity_id[predicted_tail]

    score_true = score_triple(s, p, o)
    score_pred = score_triple(s, p, p2)

    sig = get_type_constraint_signal(head, relation, true_tail, predicted_tail, constraints)
    summary = (
        f"Model predicted '{predicted_tail}' (score={score_pred:.3f}) "
        f"over '{true_tail}' (score={score_true:.3f}). "
        f"Type fit: '{true_tail}'={sig['type_fit_true']:.3f} vs "
        f"'{predicted_tail}'={sig['type_fit_pred']:.3f} for '{relation}'. "
        f"Expected tails: {', '.join(sig['expected_tails'][:3])}. "
        f"Only '{true_tail}' in: {', '.join(sig['only_true_has'][:5]) or 'none'}. "
        f"Only '{predicted_tail}' in: {', '.join(sig['only_pred_has'][:5]) or 'none'}."
    )
    return summary, {
        "score_true":     score_true,
        "score_pred":     score_pred,
        "shared":         set(sig["shared_rels"]),
        "only_true":      set(sig["only_true_has"]),
        "only_pred":      set(sig["only_pred_has"]),
        "type_fit_true":  sig["type_fit_true"],
        "type_fit_pred":  sig["type_fit_pred"],
        "type_gap":       sig["type_gap"],
        "expected_tails": sig["expected_tails"],
    }


In [71]:
@torch.no_grad()
def batch_rank_triples(tuples_batch: list, hard_threshold: int) -> list:
    valid = []
    for i, (h, r, t) in enumerate(tuples_batch):
        s = label_to_entity_id.get(h)
        p = label_to_relation_id.get(r)
        o = label_to_entity_id.get(t)
        if None in (s, p, o):
            valid.append((i, None))
        else:
            valid.append((i, (s, p, o, h, t)))

    good = [(i, d) for i, d in valid if d is not None]
    out  = [None] * len(tuples_batch)
    if not good:
        return out

    s_ids = torch.tensor([d[0] for _, d in good], device=DEVICE)
    p_ids = torch.tensor([d[1] for _, d in good], device=DEVICE)

    s_re = ENT_RE[s_ids]
    s_im = ENT_IM[s_ids]
    p_re = relation_emb_gpu[p_ids, :DIM]
    p_im = relation_emb_gpu[p_ids, DIM:]

    scores = (s_re * p_re - s_im * p_im) @ ENT_RE.T \
           + (s_re * p_im + s_im * p_re) @ ENT_IM.T

    for row_pos, (orig_idx, d) in enumerate(good):
        s, p, o, h_label, t_label = d
        row = scores[row_pos].clone()
        row[s] = float("-inf")

        ranked_ids  = torch.argsort(row, descending=True).cpu()
        scores_cpu  = row.cpu().numpy()
        ranked_list = ranked_ids.tolist()
        true_rank   = ranked_list.index(o) + 1

        out[orig_idx] = {
            "predicted":       id_to_entity_label[ranked_list[0]],
            "predicted_score": float(scores_cpu[ranked_list[0]]),
            "true_tail":       t_label,
            "true_score":      float(scores_cpu[o]),
            "true_rank":       true_rank,
            "full_ranking":    [(id_to_entity_label[i], round(float(scores_cpu[i]), 3))
                                for i in ranked_list[:200]],
            "hard_failure":    bool(true_rank > hard_threshold),
        }

    return out

In [72]:
import os, json, time
import torch
import torch.nn.functional as F
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

entity_emb_gpu   = entity_emb.to(DEVICE).float()
relation_emb_gpu = relation_emb.to(DEVICE).float()
N_ENT = entity_emb_gpu.shape[0]
DIM   = entity_emb_gpu.shape[1] // 2

ENT_RE = entity_emb_gpu[:, :DIM].contiguous()
ENT_IM = entity_emb_gpu[:, DIM:].contiguous()

_mag      = torch.sqrt(ENT_RE**2 + ENT_IM**2)
EMBS_NORM = F.normalize(_mag, dim=-1).contiguous()
EMBS_CACHE = _mag.cpu()

print("Building entity-relation cache …")
ent_to_rels: dict[str, set] = {}
for _, row in df_train.iterrows():
    h, r, t = row["head"], row["relation"], row["tail"]
    ent_to_rels.setdefault(h, set()).add(r)
    ent_to_rels.setdefault(t, set()).add(r)
print(f"  cached {len(ent_to_rels)} entities")

# Pre-cache head → relation → list of tails from training
ent_rel_to_tails: dict[tuple, list] = {}
for _, row in df_train.iterrows():
    key = (row["head"], row["relation"])
    ent_rel_to_tails.setdefault(key, []).append(row["tail"])
print(f"  cached {len(ent_rel_to_tails)} (entity, relation) pairs")

constraints = build_type_constraints(df_train)


def is_hard_failure(ranking, head, relation, constraints, hop_type="multi"):
    if ranking["true_rank"] == 1: return False
    if hop_type == "none":        return False
    return ranking["true_rank"] <= 12


def reclassify_hard(records, constraints):
    reclassified = 0
    for r in records:
        fake_ranking = {
            "true_rank":       r["true_rank"],
            "true_tail":       r["tail"],
            "predicted":       r["predicted"],
            "true_score":      r["score_true"],
            "predicted_score": r["score_predicted"],
        }
        new_hard = is_hard_failure(
            fake_ranking,
            r["head"],
            r["relation"],
            constraints,
            hop_type=r.get("hop_type", "multi")
        )
        if new_hard != r["hard_failure"]:
            r["hard_failure"] = new_hard
            reclassified += 1
    print(f"Reclassified {reclassified} records")
    return records


@torch.no_grad()
def batch_similarity_summary(entity_labels: list, k: int = 5) -> list:
    ids  = torch.tensor([label_to_entity_id[e] for e in entity_labels], device=DEVICE)
    vecs = EMBS_NORM[ids]
    sim_matrix = vecs @ EMBS_NORM.T
    topk_vals, topk_ids = torch.topk(sim_matrix, k + 1, dim=-1)
    topk_ids_cpu  = topk_ids.cpu().numpy()
    topk_vals_cpu = topk_vals.cpu().numpy()
    results = []
    for b_idx, entity in enumerate(entity_labels):
        e_rels  = ent_to_rels.get(entity, set())
        entries = []
        for j in range(topk_ids_cpu.shape[1]):
            idx  = int(topk_ids_cpu[b_idx, j])
            name = id_to_entity_label[idx]
            if name == entity:
                continue
            score  = float(topk_vals_cpu[b_idx, j])
            n_rels = ent_to_rels.get(name, set())
            shared = len(e_rels & n_rels)
            total  = len(e_rels | n_rels)
            rel_sc = shared / total if total > 0 else 0.0
            entries.append((name, score, shared, rel_sc))
            if len(entries) == k:
                break
        parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})" for n, s, sh, rs in entries]
        summary = f"{entity} most similar to: {', '.join(parts)}"
        results.append((summary, entries))
    return results


def _cpu_record(h, r, t, ranking):
    sg = focused_subgraph(
        [h, t, ranking["predicted"]],
        graph, hops=3, max_triples=150, query_relation=r,
    )
    hop_type, hops, _ = hop_classifier(h, t, graph, target_relation=r)
    fail_sum, fail_raw = failure_summary(h, r, t, ranking["predicted"], constraints)
    return sg, hop_type, int(hops), fail_sum, fail_raw


def preprocess_all_triples_gpu(df, split_name, batch_size=128, cpu_workers=4):
    records        = []
    n_entities     = len(label_to_entity_id)
    hard_threshold = max(10, int(n_entities * 0.15))
    rows           = [(r["head"], r["relation"], r["tail"]) for _, r in df.iterrows()]
    total          = len(rows)
    t0             = time.time()

    for batch_start in range(0, total, batch_size):
        batch = rows[batch_start: batch_start + batch_size]
        valid = [
            (h, r, t) for h, r, t in batch
            if h in label_to_entity_id
            and t in label_to_entity_id
            and r in label_to_relation_id
        ]
        if not valid:
            continue

        rankings = batch_rank_triples(valid, hard_threshold)

        heads = [h for h, r, t in valid]
        tails = [t for h, r, t in valid]
        sims  = batch_similarity_summary(heads + tails, k=5)
        sim_heads = sims[:len(heads)]
        sim_tails = sims[len(heads):]

        def _work(args):
            i, (h, r, t) = args
            if rankings[i] is None:
                return i, None
            try:
                return i, _cpu_record(h, r, t, rankings[i])
            except Exception as e:
                print(f"\n  [CPU ERR] {h} | {r} | {t}: {e}")
                return i, None

        cpu_out = {}
        with ThreadPoolExecutor(max_workers=cpu_workers) as ex:
            for i, res in ex.map(_work, enumerate(valid)):
                cpu_out[i] = res

        for i, (h, r, t) in enumerate(valid):
            rk  = rankings[i]
            cpu = cpu_out.get(i)
            if rk is None or cpu is None:
                continue
            sg, hop_type, hops, fail_sum, fail_raw = cpu

            records.append({
                "split":            split_name,
                "head":             h,
                "relation":         r,
                "tail":             t,
                "true_rank":        int(rk["true_rank"]),
                "predicted":        rk["predicted"],
                "score_true":       float(rk["true_score"]),
                "score_predicted":  float(rk["predicted_score"]),
                "top5":             rk["full_ranking"][:15],
                "hop_type":         hop_type,
                "hop_count":        hops,
                "sim_head":         sim_heads[i][0],
                "sim_tail":         sim_tails[i][0],
                "fail_summary":     fail_sum,
                "subgraph":         sg,
                "shared_relations": list(fail_raw["shared"]),
                "only_tail_has":    list(fail_raw["only_true"]),
                "only_pred_has":    list(fail_raw["only_pred"]),
                "hard_failure":     is_hard_failure(
                                        rk, h, r, constraints,
                                        hop_type=hop_type),
                "known_tails":      ent_rel_to_tails.get((h, r), []),
            })

        done    = min(batch_start + batch_size, total)
        elapsed = time.time() - t0
        rate    = done / elapsed if elapsed > 0 else 0
        eta     = (total - done) / rate if rate > 0 else 0
        print(f"  {split_name}: {done}/{total}  "
              f"records={len(records)}  "
              f"{rate:.1f} rows/s  ETA {eta/60:.1f}min", end="\r")

    print()
    return records


PREPROCESSED_TRAIN = "/kaggle/working/CODEX_S_preprocessed_train.json"
PREPROCESSED_TEST  = "/kaggle/working/CODEX_S_preprocessed_test.json"

for f in [PREPROCESSED_TRAIN, PREPROCESSED_TEST]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")


t0 = time.time()
print("Preprocessing train …")
train_records = preprocess_all_triples_gpu(df_train, "train", batch_size=128)
print("Preprocessing test …")
test_records  = preprocess_all_triples_gpu(df_test,  "test",  batch_size=128)
with open(PREPROCESSED_TRAIN, "w") as f: json.dump(train_records, f, indent=2)
with open(PREPROCESSED_TEST,  "w") as f: json.dump(test_records,  f, indent=2)
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

if EMBS_CACHE is None:
    EMBS_CACHE = _mag.cpu()

sample = test_records[0]
print(f"\nVerification — top5 length: {len(sample['top5'])}  ← must be 15")
print(f"true_rank of sample: {sample['true_rank']}")
print(f"Correct answer visible: {sample['true_rank'] <= len(sample['top5'])}")
print(f"known_tails sample: {sample['known_tails'][:5]}")

Device: cuda  |  GPU: Tesla T4
Building entity-relation cache …
  cached 2034 entities
  cached 10465 (entity, relation) pairs
Preprocessing train …
  train: 32888/32888  records=32888  390.2 rows/s  ETA 0.0min
Preprocessing test …
  test: 1828/1828  records=1828  385.8 rows/s  ETA 0.0min

Done in 1.8 min

Verification — top5 length: 15  ← must be 15
true_rank of sample: 1
Correct answer visible: True
known_tails sample: []


In [73]:
import json, torch, numpy as np
import torch.nn.functional as F
from collections import defaultdict

# ── config ────────────────────────────────────────────────────────────────────
TOPK           = 50
HELD_OUT_INPUT = "/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/CODEX_S_held_out (2).json"
HELD_OUT_OUT   = "/kaggle/working/codex_bert_held_out.json"

# ── All of these already exist from your earlier cells ───────────────────────
# label_to_entity_id, id_to_entity_label
# label_to_relation_id, id_to_relation_label
# entity_emb, relation_emb, DIM
# _scores_sp, score_triple
# ENT_RE, ENT_IM, EMBS_NORM
# constraints (from build_type_constraints)
# ent_rel_to_tails, ent_to_rels
# df_train

# ── Entity relation profile (same as UMLS version) ───────────────────────────
# ent_to_rels already built in your GPU setup cell
# use it directly — no rebuild needed

# ── EMBS for cosine similarity ────────────────────────────────────────────────
# EMBS_CACHE already built — use it directly

print(f"Entities:  {len(label_to_entity_id)}")
print(f"Relations: {len(label_to_relation_id)}")
print(f"EMBS:      {EMBS_CACHE.shape}")




@torch.no_grad()
def get_topk_candidates_codex(head, relation, true_tail, k=TOPK):
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    o = label_to_entity_id.get(true_tail)

    if None in (s, p):
        return []

    # Score all entities at once on GPU
    scores_gpu = (
        (ENT_RE[s] * relation_emb_gpu[p, :DIM] -
         ENT_IM[s] * relation_emb_gpu[p, DIM:]) @ ENT_RE.T +
        (ENT_RE[s] * relation_emb_gpu[p, DIM:] +
         ENT_IM[s] * relation_emb_gpu[p, :DIM]) @ ENT_IM.T
    )
    scores_gpu[s] = float("-inf")   # exclude self
    scores_cpu = scores_gpu.cpu().numpy()

    ranked_ids = np.argsort(scores_cpu)[::-1]

    candidates = []
    rank = 0
    for idx in ranked_ids:
        entity = id_to_entity_label[idx]
        if entity == head:
            continue
        rank += 1
        candidates.append({
            "entity":    entity,
            "kge_score": round(float(scores_cpu[idx]), 4),
            "kge_rank":  rank,
        })
        if len(candidates) == k:
            break

    # force-insert true tail if outside top-K
    if true_tail and not any(c["entity"] == true_tail for c in candidates):
        if o is not None:
            t_score = float(scores_cpu[o])
            t_rank  = int(np.sum(scores_cpu > t_score))
            candidates.append({
                "entity":    true_tail,
                "kge_score": round(t_score, 4),
                "kge_rank":  t_rank,
            })

    return candidates



def extract_features_codex(head, relation, candidate, true_tail):
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    c = label_to_entity_id.get(candidate)

    if None in (s, p, c):
        return {}

    # KGE score
    kge_score = score_triple(s, p, c)

    # Embedding similarity using magnitude embeddings
    h_vec = EMBS_CACHE[s]
    c_vec = EMBS_CACHE[c]
    sim_to_head = F.cosine_similarity(
        h_vec.unsqueeze(0), c_vec.unsqueeze(0)
    ).item()

    o = label_to_entity_id.get(true_tail)
    if o is not None:
        t_vec = EMBS_CACHE[o]
        sim_to_true_tail = F.cosine_similarity(
            t_vec.unsqueeze(0), c_vec.unsqueeze(0)
        ).item()
    else:
        sim_to_true_tail = 0.0

    # Relational profile overlap
    # ent_to_rels maps entity → set of relations it participates in
    true_rels = ent_to_rels.get(true_tail, set())
    cand_rels = ent_to_rels.get(candidate, set())

    shared    = true_rels & cand_rels
    only_true = true_rels - cand_rels
    only_cand = cand_rels - true_rels
    union     = true_rels | cand_rels
    jaccard   = len(shared) / len(union) if union else 0.0

    # Type constraint signals
    rel_dist  = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = rel_dist.get(candidate, 0.0)
    ranked    = list(rel_dist.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    cooc      = constraints["rel_to_tail_counts"].get(
                    relation, {}).get(candidate, 0)

    # In training signal  (CoDEx-S specific — not in UMLS version)
    known_as_tail = candidate in ent_rel_to_tails.get((head, relation), [])

    return {
        # Nations/UMLS compatible keys
        "kge_score":         round(kge_score,        4),
        "sim_to_head":       round(sim_to_head,       4),
        "sim_to_true_tail":  round(sim_to_true_tail,  4),
        "shared_rels_count": len(shared),
        "only_true_count":   len(only_true),
        "only_cand_count":   len(only_cand),
        "jaccard_overlap":   round(jaccard,           4),
        "direct_edge":       int(tf > 0.0),
        "hop_count":         1,
        # CoDEx-S extras
        "type_fit":          round(tf,                4),
        "type_rank":         type_rank,
        "cooc_count":        cooc,
        "in_training":       int(known_as_tail),      # ← key CoDEx-S feature
    }



def build_text_input_codex(head, relation, candidate,
                            features, true_tail=None):
    f         = features
    sig       = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = sig.get(candidate, 0.0)
    ranked    = list(sig.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    expected  = ", ".join(ranked[:3]) if ranked else "none"
    cand_rels = sorted(ent_to_rels.get(candidate, set()))[:5]

    # Known tails for this (head, relation) pair
    known = ent_rel_to_tails.get((head, relation), [])
    known_str = ", ".join(known[:5]) if known else "none"

    text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{candidate}\n\n"
        f"[TYPE CONSTRAINTS]\n"
        f"type_fit = {tf:.4f}\n"
        f"type_rank = {type_rank}\n"
        f"expected_tails = {expected}\n\n"
        f"[KGE SIGNALS]\n"
        f"kge_score = {f.get('kge_score', 0.0):.3f}\n\n"
        f"[TRAINING EVIDENCE]\n"
        f"known_tails = {known_str}\n"
        f"in_training = {f.get('in_training', 0)}\n\n"
        f"[RELATIONAL PROFILE]\n"
        f"candidate_appears_in = "
        f"{', '.join(cand_rels) if cand_rels else 'none'}\n"
    )

    if true_tail and true_tail != candidate:
        only_true = sorted(
            ent_to_rels.get(true_tail, set()) -
            ent_to_rels.get(candidate, set())
        )[:5]
        text += (
            f"\n[DIFFERENCE]\n"
            f"only_true_has = "
            f"{' | '.join(only_true) if only_true else 'none'}\n"
        )

    return text



def convert_held_out_record_codex(record):
    head      = record["head"]
    relation  = record["relation"]
    true_tail = record["tail"]

    if head      not in label_to_entity_id:   return None
    if true_tail not in label_to_entity_id:   return None
    if relation  not in label_to_relation_id: return None

    # Get top-K candidates from ComplEx
    raw = get_topk_candidates_codex(head, relation, true_tail, k=TOPK)
    if not raw:
        return None

    # Reuse precomputed fields from preprocessed record
    only_tail_has = record.get("only_tail_has", [])
    fail_summary  = record.get("fail_summary",  "")
    sim_head_str  = record.get("sim_head",       "")
    known_tails   = record.get("known_tails",    [])

    # Subgraph string
    raw_subgraph = record.get("subgraph", [])
    oth_set      = set(only_tail_has)
    subgraph_str = "\n".join(
        f"  {h} --{r}--> {t}" + (" ◆" if r in oth_set else "")
        for triple in raw_subgraph[:12]
        for h, r, t in [triple if isinstance(triple, (list, tuple))
                         else (triple.get("head",""),
                               triple.get("relation",""),
                               triple.get("tail",""))]
    )

    # Build candidates with features + text_input
    candidates_out = []
    for c in raw:
        entity   = c["entity"]
        features = extract_features_codex(head, relation, entity, true_tail)
        if not features:
            continue
        text_input = build_text_input_codex(
            head, relation, entity,
            features  = features,
            true_tail = true_tail if entity != true_tail else None
        )
        candidates_out.append({
            "entity":     entity,
            "label":      1 if entity == true_tail else 0,
            "soft_label": None,
            "neg_type":   "true" if entity == true_tail else "model_topk",
            "kge_rank":   c["kge_rank"],
            "text_input": text_input,
            "features":   features,
        })

    # Sort by kge_score descending
    candidates_out.sort(
        key=lambda c: -(c["features"].get("kge_score") or -99)
    )

    rotate_rank = next(
        (i + 1 for i, c in enumerate(candidates_out)
         if c["entity"] == true_tail), None
    )

    return {
        "triple":              [head, relation, true_tail],
        "true_rank":           record.get("true_rank", -1),
        "hop_type":            record.get("hop_type",  "multi"),
        "hard_failure":        record.get("hard_failure", False),
        "agent_confidence":    0.5,
        "rotate_rank_in_topk": rotate_rank,
        "candidates":          candidates_out,
        "context": {
            "subgraph_str":  subgraph_str,
            "only_tail_has": only_tail_has,
            "fail_summary":  fail_summary,
            "sim_head":      sim_head_str,
            "known_tails":   known_tails,   # ← CoDEx-S extra
        }
    }



with open(HELD_OUT_INPUT) as f:
    held_out_records = json.load(f)

print(f"Total held-out: {len(held_out_records)}")

reranker_held_out, errors = [], 0

for i, record in enumerate(held_out_records):
    try:
        out = convert_held_out_record_codex(record)
        if out is not None:
            reranker_held_out.append(out)
    except Exception as e:
        print(f"[ERROR] {i}: {e}")
        errors += 1
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(held_out_records)} done")

with open(HELD_OUT_OUT, "w") as f:
    json.dump(reranker_held_out, f, indent=2)

print(f"\nOutput: {len(reranker_held_out)} records  ({errors} errors)")

# ── Sanity checks ─────────────────────────────────────────────────────────────
missing = sum(
    1 for r in reranker_held_out
    if not any(c["entity"] == r["triple"][2] for c in r["candidates"])
)
print(f"Missing true tail in candidates: {missing}")

rotate_ranks = [
    r["rotate_rank_in_topk"]
    for r in reranker_held_out
    if r["rotate_rank_in_topk"] is not None
]
hits1 = sum(1 for r in rotate_ranks if r == 1)
hits3 = sum(1 for r in rotate_ranks if r <= 3)

print(f"\nComplEx baseline within top-{TOPK}:")
print(f"  Hits@1: {hits1}/{len(rotate_ranks)}"
      f"  ({100*hits1/max(len(rotate_ranks),1):.1f}%)")
print(f"  Hits@3: {hits3}/{len(rotate_ranks)}"
      f"  ({100*hits3/max(len(rotate_ranks),1):.1f}%)")

# Type constraint coverage
has_type_fit  = sum(
    1 for r in reranker_held_out
    for c in r["candidates"]
    if c["features"].get("type_fit", 0) > 0
)
in_training   = sum(
    1 for r in reranker_held_out
    for c in r["candidates"]
    if c["features"].get("in_training", 0) == 1
)
total_cands   = sum(len(r["candidates"]) for r in reranker_held_out)

print(f"\nFeature coverage:")
print(f"  type_fit > 0:  {has_type_fit}/{total_cands}"
      f"  ({100*has_type_fit/max(total_cands,1):.1f}%)")
print(f"  in_training=1: {in_training}/{total_cands}"
      f"  ({100*in_training/max(total_cands,1):.1f}%)")

print(f"\nSample positive text_input:")
pos = next(c for c in reranker_held_out[0]["candidates"] if c["label"] == 1)
print(pos["text_input"])

print(f"\nSample negative text_input:")
neg = next(c for c in reranker_held_out[0]["candidates"] if c["label"] == 0)
print(neg["text_input"])

print(f"\nSaved → {HELD_OUT_OUT}")

Entities:  2034
Relations: 42
EMBS:      torch.Size([2034, 256])
Total held-out: 914
  25/914 done
  50/914 done
  75/914 done
  100/914 done
  125/914 done
  150/914 done
  175/914 done
  200/914 done
  225/914 done
  250/914 done
  275/914 done
  300/914 done
  325/914 done
  350/914 done
  375/914 done
  400/914 done
  425/914 done
  450/914 done
  475/914 done
  500/914 done
  525/914 done
  550/914 done
  575/914 done
  600/914 done
  625/914 done
  650/914 done
  675/914 done
  700/914 done
  725/914 done
  750/914 done
  775/914 done
  800/914 done
  825/914 done
  850/914 done
  875/914 done
  900/914 done

Output: 914 records  (0 errors)
Missing true tail in candidates: 0

ComplEx baseline within top-50:
  Hits@1: 114/914  (12.5%)
  Hits@3: 215/914  (23.5%)

Feature coverage:
  type_fit > 0:  35961/45789  (78.5%)
  in_training=1: 9266/45789  (20.2%)

Sample positive text_input:
[QUERY]
Samoa | member of | ?

[CANDIDATE]
"African, Caribbean and Pacific Group of States"

[TYPE C